# Analisis Penjualan & Efisiensi Logistik E-Commerce Indonesia (2023–2025)
**Studi Empiris 20.848 Transaksi, Risiko Pembatalan COD & Efisiensi Subsidi Ongkir**

> **Author:** Ferdian Hanif  
> **Role:** Data Analyst  
> **Email:** ferdianhanif001@gmail.com  
> **Lokasi:** Jakarta Barat, Indonesia  
> **Laporan Presentasi Eksekutif:** [presentation_deck.pdf](../reports/presentation_deck.pdf)

---

## Ringkasan Eksekutif

Proyek ini menganalisis **20.848 pesanan e-commerce** selama 24 bulan masa operasional (Desember 2023 hingga November 2025). Fokus utama analisis adalah membedah dinamika penjualan toko online, risiko kegagalan transaksi pada metode *Cash on Delivery* (COD), serta disparitas biaya logistik dan beban subsidi ongkos kirim antarpulau.

### Tiga Temuan Kunci:
1. **Risiko & Ketergantungan Metode COD:** Metode COD menyumbang **55,3% total transaksi toko (11.538 pesanan)** dengan rasio pembatalan 13,4% (1.544 pesanan gagal/dibatalkan). Sebaliknya, pembayaran digital via *ShopeePay* mencatat tingkat pembatalan terendah (8,7%), sedangkan pembayaran gerai retail minimarket mencatat tingkat kegagalan ekstrem (48,0%).
2. **Disparitas Logistik Jawa vs Luar Jawa:** Pesanan ke luar Pulau Jawa menanggung rata-rata ongkir **Rp 37.096 per paket (naik 2,7 kali lipat)** dibanding Pulau Jawa (Rp 13.544). Mahalnya ongkir ini memicu lonjakan tingkat pembatalan pesanan di luar Jawa hingga 18,9%.
3. **Beban Subsidi Ongkos Kirim:** Dari total tagihan riil kurir sebesar **Rp 384,1 Juta**, subsidi platform dan toko menanggung **58,2% (Rp 223,5 Juta)**, sementara pembeli hanya membayar 22,7% (Rp 87,4 Juta). Sebanyak 69,8% transaksi dinikmati pembeli dengan fasilitas Bebas Ongkir Rp 0.

---

## Bab 1: Konteks Bisnis & Tujuan Analisis

### 1.1 Masalah Bisnis
Pertumbuhan pesat e-commerce di Indonesia diiringi dengan tantangan operasional logistik kepulauan dan friksi sistem pembayaran. Toko e-commerce sering kali menghadapi:
- **Tingginya Retur Paket COD:** Pesanan COD yang ditolak saat kurir tiba (*Return to Origin / RTO*) menimbulkan biaya pengemasan sia-sia dan kerusakan stok produk.
- **Erosi Margin Akibat Subsidi Ongkir:** Promosi bebas ongkir berhasil memacu volume, namun dapat menggerus margin laba kotor jika tidak diimbangi dengan batas belanja minimum yang rasional, terutama untuk pengiriman ke luar Pulau Jawa.

### 1.2 Pertanyaan Analisis
1. Bagaimana tren pertumbuhan volume transaksi dan omzet bulanan selama periode 2023–2025?
2. Bagaimana perbandingan rasio pembatalan transaksi antarmetode pembayaran (COD vs E-Wallet vs Transfer Bank)?
3. Seberapa besar disparitas ongkos kirim dan tingkat pembatalan antara pembeli di Pulau Jawa dan Luar Pulau Jawa?
4. Bagaimana struktur pembebanan biaya pengiriman antara pembeli, platform, dan toko?

---

## Bab 2: Audit Integritas Data (Data Hygiene & Cleaning)

Pada bab ini, kita memuat dataset mentah, melakukan audit kualitas data (*missing values*, duplikasi, tipe data), menginferensikan cap waktu yang hilang dari metadata berkas bulanan, dan memodelkan data ke dalam format relasional SQLite.

In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setup direktori dan konfigurasi plotting
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
CLEAN_CSV_PATH = os.path.join(BASE_DIR, "data", "processed", "clean_ecommerce_sales.csv")
DB_PATH = os.path.join(BASE_DIR, "data", "processed", "ecommerce_analytics.db")

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

print("Memuat dataset yang telah dibersihkan...")
df = pd.read_csv(CLEAN_CSV_PATH, low_memory=False)
print(f"Total baris data: {len(df):,}")
print(f"Total kolom: {len(df.columns)}")
df.head(3)

In [ ]:
# Pengecekan Integritas Tipe Data dan Missing Values
print("Ringkasan Integritas Kolom Kunci:")
audit_summary = pd.DataFrame({
    'Tipe Data': df.dtypes,
    'Nilai Kosong': df.isnull().sum(),
    'Persentase Kosong (%)': (df.isnull().sum() / len(df)) * 100
})
audit_summary

---

## Bab 3: Exploratory Data Analysis (EDA)

### 3.1 Ringkasan Metrik Makro Portofolio
Kita menghitung metrik agregat utama: total volume, omzet kotor, tingkat pesanan selesai, dan rasio adopsi bebas ongkir.

In [ ]:
total_orders = len(df)
completed_orders = (df['status_kategori'] == 'Selesai').sum()
cancelled_orders = (df['status_kategori'] == 'Batal').sum()
total_revenue = df['total_payment'].sum()
total_units = df['order_qty'].sum()
free_ship_orders = (df['is_free_shipping'] == 1).sum()

print("=== RINGKASAN METRIK MAKRO (2023 - 2025) ===")
print(f"Total Pesanan Masuk       : {total_orders:,}")
print(f"Pesanan Selesai (Sukses)  : {completed_orders:,} ({(completed_orders/total_orders)*100:.1f}%)")
print(f"Pesanan Dibatalkan / Gagal: {cancelled_orders:,} ({(cancelled_orders/total_orders)*100:.1f}%)")
print(f"Total Nilai Transaksi     : Rp {total_revenue:,.0f}")
print(f"Total Unit Barang Terjual : {total_units:,} unit")
print(f"Adopsi Bebas Ongkir (Rp 0): {free_ship_orders:,} ({(free_ship_orders/total_orders)*100:.1f}%)")

### 3.2 Analisis Tren Penjualan Bulanan (Run-Rate)
Menganalisis fluktuasi jumlah pesanan dan perputaran omzet bulanan dari Desember 2023 hingga November 2025.

In [ ]:
monthly_trend = df.groupby('year_month').agg(
    total_pesanan=('order_id', 'count'),
    total_unit=('order_qty', 'sum'),
    total_omzet=('total_payment', 'sum'),
    pesanan_batal=('is_cancelled', 'sum')
).reset_index()

monthly_trend['rasio_batal_pct'] = (monthly_trend['pesanan_batal'] / monthly_trend['total_pesanan']) * 100
monthly_trend['omzet_juta_rp'] = monthly_trend['total_omzet'] / 1e6

fig, ax1 = plt.subplots(figsize=(12, 5))
x = np.arange(len(monthly_trend))
ax1.bar(x, monthly_trend['total_pesanan'], color='#2563EB', alpha=0.85, width=0.6, label='Volume Pesanan')
ax1.set_ylabel('Volume Pesanan per Bulan', color='#2563EB', fontsize=11, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(monthly_trend['year_month'], rotation=45, ha='right', fontsize=9)
ax1.set_xlabel('Periode Transaksi', fontsize=11, fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(x, monthly_trend['omzet_juta_rp'], color='#059669', marker='o', linewidth=2.5, label='Omzet (Juta Rp)')
ax2.set_ylabel('Nilai Transaksi (Juta Rp)', color='#059669', fontsize=11, fontweight='bold')
ax2.grid(False)

plt.title('Tren Penjualan Bulanan & Nilai Transaksi (Desember 2023 - November 2025)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

---

## Bab 4: Temuan Akar Masalah (Root Cause Findings)

### 4.1 Hubungan Metode Pembayaran dengan Risiko Pembatalan
Metode pembayaran menjadi salah satu penentu komitmen beli konsumen. Kita membandingkan pangsa volume transaksi terhadap rasio pembatalan masing-masing metode.

In [ ]:
payment_eval = df.groupby('payment_group').agg(
    total_pesanan=('order_id', 'count'),
    pesanan_batal=('is_cancelled', 'sum'),
    total_omzet=('total_payment', 'sum'),
    rata_rata_order=('total_payment', 'mean')
).reset_index()

payment_eval['pangsa_pesanan_pct'] = (payment_eval['total_pesanan'] / len(df)) * 100
payment_eval['rasio_batal_pct'] = (payment_eval['pesanan_batal'] / payment_eval['total_pesanan']) * 100
payment_eval = payment_eval.sort_values('total_pesanan', ascending=False)

print(payment_eval.to_string(index=False))

### 4.2 Disparitas Logistik Jawa vs Luar Jawa
Menganalisis beban biaya pengiriman riil yang ditagihkan kurir dan tingkat pembatalan pesanan berdasarkan klaster geografis.

In [ ]:
regional_eval = df.groupby('region_group').agg(
    total_pesanan=('order_id', 'count'),
    total_omzet=('total_payment', 'sum'),
    rata_rata_ongkir=('shipping_actual_cost', 'mean'),
    rata_rata_subsidi=('shipping_subsidy', 'mean'),
    rata_rata_ongkir_pembeli=('shipping_buyer_paid', 'mean'),
    rasio_batal_pct=('is_cancelled', 'mean')
).reset_index()

regional_eval['pangsa_pesanan_pct'] = (regional_eval['total_pesanan'] / len(df)) * 100
regional_eval['rasio_batal_pct'] = regional_eval['rasio_batal_pct'] * 100
print(regional_eval.to_string(index=False))

### 4.3 Struktur Pembebanan Biaya Logistik (Siapa yang Menanggung Ongkir?)
Menghitung proporsi beban pengiriman riil kurir yang ditanggung oleh pembeli versus subsidi platform/toko.

In [ ]:
total_actual = df['shipping_actual_cost'].sum()
total_buyer = df['shipping_buyer_paid'].sum()
total_subsidy = df['shipping_subsidy'].sum()
total_gap = max(0, total_actual - (total_buyer + total_subsidy))

print("=== STRUKTUR BEBAN LOGISTIK NASIONAL ===")
print(f"Total Biaya Riil Jasa Kurir : Rp {total_actual:,.0f} (100.0%)")
print(f"Beban Ditanggung Pembeli    : Rp {total_buyer:,.0f} ({(total_buyer/total_actual)*100:.1f}%)")
print(f"Subsidi Promo Platform/Toko : Rp {total_subsidy:,.0f} ({(total_subsidy/total_actual)*100:.1f}%)")
print(f"Selisih Beban Toko / Gap    : Rp {total_gap:,.0f} ({(total_gap/total_actual)*100:.1f}%)")

---

## Bab 5: Rekomendasi Taktis & Estimasi Dampak Bisnis

Berdasarkan temuan data di atas, berikut adalah rekomendasi strategis yang dapat dieksekusi tim operasional dan manajemen komersial:

### 5.1 Rekomendasi Pengendalian Risiko COD & Metode Pembayaran
1. **Filter Transaksi COD Antarpulau:** Nonaktifkan opsi COD untuk pengiriman luar Pulau Jawa dengan nilai pesanan di atas Rp 200.000 atau pembeli baru tanpa reputasi. Langkah ini dapat menekan risiko paket gagal kirim (*Return to Origin / RTO*) yang memakan ongkir hingga Rp 37.000 per paket.
2. **Sistem Konfirmasi Pesanan Instan:** Kirimkan notifikasi verifikasi via WhatsApp dalam 15 menit pertama setelah pesanan COD dibuat untuk memangkas pembatalan spontan (*berubah pikiran*).
3. **Penutupan Gerai Minimarket:** Hentikan opsi pembayaran via kasir Indomaret/Alfamart yang mencatat tingkat kegagalan pembayaran hingga 48,0% dan menahan stok barang secara sia-sia selama 24 jam.

### 5.2 Rekomendasi Efisiensi Logistik & Subsidi Ongkos Kirim
1. **Penyesuaian Batas Belanja Luar Jawa:** Naikkan syarat minimum belanja bebas ongkir dari Rp 30.000 menjadi Rp 60.000 khusus pengiriman ke luar Pulau Jawa agar nilai margin kotor mampu menutup selisih biaya logistik.
2. **Standarisasi Bobot Packing:** Terapkan kemasan kardus ringan di bawah 1.000 gram agar produk tidak melonjak ke tier tarif 2 kg pada jasa kurir hemat dan reguler.

### 5.3 Proyeksi Dampak Finansial
- Penurunan rasio pembatalan COD dari 13,4% menjadi di bawah 9,0% diperkirakan menyelamatkan potensi omzet sebesar **Rp 18,2 Juta per tahun**.
- Penyesuaian minimum belanja luar pulau diproyeksikan menghemat beban subsidi ongkir sebesar **15% - 20% (setara Rp 11 Juta - Rp 14 Juta per tahun)** tanpa menurunkan volume transaksi pembeli inti.